In [1]:
#%load_ext cudf.pandas
#%load_ext cuml.accel

import cudf
import pandas as pd
import cupy as cp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter
import tqdm
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [2]:
df = pd.read_csv('data/02_preprocessed_data.csv')
print(type(df))
df.sample(3)

<class 'pandas.core.frame.DataFrame'>


,question1,question2,is_duplicate
155078,i got engaged months before and now get to kno...,my fiancee broke up with me months ago i have ...,0
274945,how can i gain my puppy a trust back after hit...,why is my puppy screaming every tears in my ey...,0
371650,why if i divide any number by do i get percent,how do you find out what out of is in terms of...,0


In [3]:
df['is_duplicate'].to_csv('data/03_y.csv', index=False)
df.drop(columns=["is_duplicate"], inplace=True)

In [4]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

In [5]:
df['tok_q1'] = df['question1'].apply(tokenize)
df['tok_q2'] = df['question2'].apply(tokenize)

In [6]:
df.head(3)

,question1,question2,tok_q1,tok_q2
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,"[what, is, the, step, by, step, guide, to, inv...","[what, is, the, step, by, step, guide, to, inv..."
1,what is the story of kohinoor kos i door diamond,what would happen if the indian government sto...,"[what, is, the, story, of, kohinoor, kos, i, d...","[what, would, happen, if, the, indian, governm..."
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,"[how, can, i, increase, the, speed, of, my, in...","[how, can, internet, speed, be, increased, by,..."


In [7]:
all_sentences = df['tok_q1'].tolist() + df['tok_q2'].tolist()

In [8]:
print(all_sentences[:5])

[['what', 'is', 'the', 'step', 'by', 'step', 'guide', 'to', 'invest', 'in', 'share', 'market', 'in', 'india'], ['what', 'is', 'the', 'story', 'of', 'kohinoor', 'kos', 'i', 'door', 'diamond'], ['how', 'can', 'i', 'increase', 'the', 'speed', 'of', 'my', 'internet', 'connection', 'while', 'using', 'a', 'van'], ['why', 'am', 'i', 'mentally', 'very', 'lonely', 'how', 'can', 'i', 'solve', 'it'], ['which', 'one', 'dissolve', 'in', 'water', 'quickly', 'sugar', 'salt', 'methane', 'and', 'carbon', 'dioxide']]


In [9]:
counter = Counter([word for sent in all_sentences for word in sent])

In [10]:
i = 0
for word, count in counter.items():
    if i >= 10:
        break
    print(f"{word}: {count}")
    i += 1

what: 327682
is: 304192
the: 381699
step: 777
by: 19302
guide: 329
to: 212586
invest: 1694
in: 218135
share: 1356


In [11]:
counter['how']

220931

In [12]:
vocab = ['<pad>','<unk>'] + [word for word, count in counter.items()]

In [13]:
print(vocab[:100])

['<pad>', '<unk>', 'what', 'is', 'the', 'step', 'by', 'guide', 'to', 'invest', 'in', 'share', 'market', 'india', 'story', 'of', 'kohinoor', 'kos', 'i', 'door', 'diamond', 'how', 'can', 'increase', 'speed', 'my', 'internet', 'connection', 'while', 'using', 'a', 'van', 'why', 'am', 'mentally', 'very', 'lonely', 'solve', 'it', 'which', 'one', 'dissolve', 'water', 'quickly', 'sugar', 'salt', 'methane', 'and', 'carbon', 'dioxide', 'astrology', 'capricorn', 'sun', 'cap', 'moon', 'rising', 'does', 'that', 'say', 'about', 'me', 'should', 'buy', 'iago', 'be', 'good', 'geologist', 'when', 'do', 'you', 'use', 'instead', 'motorola', 'company', 'hack', 'charter', 'dix', 'method', 'find', 'separation', 'slits', 'fresnel', 'baptism', 'read', 'youtube', 'comments', 'make', 'physics', 'easy', 'learn', 'was', 'your', 'first', 'sexual', 'experience', 'like', 'are', 'laws', 'change', 'status']


In [14]:
word2idx = {w:i for i,w in enumerate(vocab)}

In [15]:
i = 0
for word, idx in word2idx.items():
    if i >= 10:
        break
    print(f"{word}: {idx}")
    i += 1

<pad>: 0
<unk>: 1
what: 2
is: 3
the: 4
step: 5
by: 6
guide: 7
to: 8
invest: 9


In [16]:
idx2word = {i:w for w,i in word2idx.items()}

In [17]:
i = 0
for idx, word in idx2word.items():
    if i >= 10:
        break
    print(f"{idx}: {word}")
    i += 1

0: <pad>
1: <unk>
2: what
3: is
4: the
5: step
6: by
7: guide
8: to
9: invest


In [18]:
vocab_size = len(vocab)
vocab_size

44729

In [19]:
enc_sentences = [[word2idx.get(word, 1) for word in sent if word in word2idx] for sent in all_sentences]
enc_sentences[:5]

[[2, 3, 4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 10, 13],
 [2, 3, 4, 14, 15, 16, 17, 18, 19, 20],
 [21, 22, 18, 23, 4, 24, 15, 25, 26, 27, 28, 29, 30, 31],
 [32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38],
 [39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 49]]

In [20]:
MAX_LEN = 40
WINDOW_SIZE = 3    #3 words before, 3 words after the central word
NEG_SAMPLES = 5

EMB_DIM = 128
BATCH_SIZE = 1024
EPOCHS = 20

In [21]:
word_freq = np.array([counter[idx2word[i]] for i in range(vocab_size)])
word_freq[:10]

array([     0,      0, 327682, 304192, 381699,    777,  19302,    329,
       212586,   1694])

In [22]:
neg_sampling_probs = word_freq ** 0.75
neg_sampling_probs[:10]

array([    0.        ,     0.        , 13695.86414027, 12952.7159367 ,
       15356.44775728,   147.16880522,  1637.57701318,    77.24972169,
        9900.36019227,   264.04932067])

In [23]:
neg_sampling_probs = neg_sampling_probs / neg_sampling_probs.sum()
neg_sampling_probs[:10]

array([0.00000000e+00, 0.00000000e+00, 1.21725342e-02, 1.15120431e-02,
       1.36484185e-02, 1.30799875e-04, 1.45543662e-03, 6.86575793e-05,
       8.79918724e-03, 2.34680291e-04])

In [24]:
def generate_pairs(enc_sentences, window_size):
    pairs = []
    for enc_sent in enc_sentences:
        sent_len = len(enc_sent)
        for idx, cen_word in enumerate(enc_sent):
            lo = max(0, idx - window_size)
            hi = min(sent_len, idx + window_size + 1)
            for j in range(lo, hi):
                if j != idx:
                    pairs.append((cen_word, enc_sent[j]))
    return pairs

In [25]:
pairs = generate_pairs(enc_sentences, window_size=WINDOW_SIZE)

In [26]:
pairs[:10]

[(2, 3),
 (2, 4),
 (2, 5),
 (3, 2),
 (3, 4),
 (3, 5),
 (3, 6),
 (4, 2),
 (4, 3),
 (4, 5)]

In [27]:
num_of_pairs = len(pairs)
num_of_pairs

44656350

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [29]:
class SkipGramCustomDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        return self.pairs[idx]

In [30]:
class SkipGramNeg(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, emb_dim)
        self.out_embed = nn.Embedding(vocab_size, emb_dim)
        nn.init.uniform_(self.in_embed.weight, -0.5/emb_dim, 0.5/emb_dim)
        nn.init.uniform_(self.out_embed.weight, -0.5/emb_dim, 0.5/emb_dim)

    def forward(self, center, context, neg_samples):
        v_center = self.in_embed(center)
        v_out = self.out_embed(context)
        v_neg = self.out_embed(neg_samples)

        pos_score = torch.sum(v_center * v_out, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(v_neg, v_center.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(1)
        
        return -(pos_loss + neg_loss).mean()

In [31]:
model = SkipGramNeg(vocab_size, EMB_DIM).to(DEVICE)

In [32]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [33]:
dataset = SkipGramCustomDataset(pairs)

In [34]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

In [35]:
neg_prob_tensor = torch.tensor(neg_sampling_probs, dtype=torch.float)

In [36]:
for epoch in range(EPOCHS):
    total_loss = 0
    for center, context in tqdm(loader, desc="Loading Dataset..."):
        center = center.to(DEVICE)
        context = context.to(DEVICE)
        neg_samples = torch.multinomial(neg_prob_tensor, center.size(0) * NEG_SAMPLES, replacement=True)
        neg_samples = neg_samples.view(center.size(0), NEG_SAMPLES).to(DEVICE)

        loss = model(center, context, neg_samples)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Loading Dataset...:   0%|          | 0/43610 [00:00<?, ?it/s]

Epoch 1, Loss: 2.1169


Loading Dataset...:   0%|          | 0/43610 [00:00<?, ?it/s]

Epoch 2, Loss: 2.0584


Loading Dataset...:   0%|          | 0/43610 [00:00<?, ?it/s]

...

Epoch 18, Loss: 2.0341


Loading Dataset...:   0%|          | 0/43610 [00:00<?, ?it/s]

Epoch 19, Loss: 2.0347


Loading Dataset...:   0%|          | 0/43610 [00:00<?, ?it/s]

Epoch 20, Loss: 2.0354


In [38]:
torch.save(model, 'word_pair_emb_model.pth')

In [39]:
embedding_matrix = model.in_embed.weight.detach()
embedding_np = embedding_matrix.cpu().numpy()
embedding_df = pd.DataFrame(embedding_np,
                           columns=[f"dim_{i}" for i in range(embedding_np.shape[1])])
embedding_df.insert(0, 'word', vocab)
embedding_df.to_csv('word_pair_emb_matrix.csv', index=False)